In [ ]:
import os
import torch
from torchvision.ops import nms
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
import cv2 # OpenCV for resizing heatmaps and polygon filling
from medvqa.metrics.bbox.utils import cxcywh_to_xyxy_tensor

from IPython.display import display, HTML

import numpy as np

from medvqa.utils.bbox_utils import convert_bboxes_into_presence_map
from medvqa.metrics.bbox.utils import compute_bbox_union_iou
from medvqa.utils.files_utils import load_pickle, load_json, load_jsonl
from medvqa.utils.metrics_utils import (
    calculate_cnr, 
    calculate_dice, 
    calculate_soft_dice, 
    calculate_segmentation_precision,
    calculate_segmentation_recall, 
    calculate_rand_index, 
    calculate_segmentation_iou
)
from medvqa.utils.constants import VINBIG_LABELS, VINBIG_LABEL2PHRASE
from medvqa.datasets.chestxdet.chestxdet_phrase_grounding_dataset_management import polygons_to_mask


# --- Helper Function ---

def get_filename_without_extension(path):
    return os.path.splitext(os.path.basename(path))[0]


# --- Main Comparator Class ---

class PhraseGroundingComparator:
    """
    Loads and compares phrase grounding predictions from three models:
    1. A custom model ("My Model")
    2. MAIRA-2
    3. BioViL-T
    """

    def __init__(
        self,
        my_model_preds_path,
        my_model_metrics_path,
        maira2_preds_path,
        biovilt_preds_path,
        using_vindrcxr=False,
        sort_indices=True,
        use_gt_polygons=False,
    ):
        print("Loading and preprocessing data...")
        self.using_vindrcxr = using_vindrcxr
        self.use_gt_polygons = use_gt_polygons
        self._load_and_preprocess_data(
            my_model_preds_path,
            my_model_metrics_path,
            maira2_preds_path,
            biovilt_preds_path,
            sort_indices,
        )
        print("Initialization complete.")

    def _load_and_preprocess_data(
        self,
        my_model_preds_path,
        my_model_metrics_path,
        maira2_preds_path,
        biovilt_preds_path,
        sort_indices,
    ):
        my_model_data = load_pickle(my_model_preds_path)
        my_model_metrics = load_pickle(my_model_metrics_path)
        self.my_model_preds = my_model_data['test_preds_and_gt']
        self.my_model_metrics = my_model_metrics['test_metrics']
        self.my_model_bbox_format = my_model_data.get('bbox_format', 'cxcywh')
        if sort_indices:
            self.my_model_sorted_idxs = np.argsort(self.my_model_metrics['soft_dices'])[::-1]
        else:
            self.my_model_sorted_idxs = list(range(len(self.my_model_metrics['soft_dices'])))

        if maira2_preds_path.endswith('.json'):
            maira2_data = load_json(maira2_preds_path)
        else:
            maira2_data = load_jsonl(maira2_preds_path)
        if self.using_vindrcxr:
            self.maira2_map = {(get_filename_without_extension(item['image_path']), VINBIG_LABEL2PHRASE[item['phrase']]): item for item in maira2_data}
        else:
            self.maira2_map = {(get_filename_without_extension(item['image_path']), item['phrase']): item for item in maira2_data}

        biovilt_data = load_pickle(biovilt_preds_path)
        if self.using_vindrcxr:
            self.biovilt_map = {(get_filename_without_extension(item['image_path']), VINBIG_LABEL2PHRASE[item['phrase']]): item for item in biovilt_data if item['split'] == 'test'}
        else:
            self.biovilt_map = {(get_filename_without_extension(item['image_path']), item['phrase']): item for item in biovilt_data if item['split'] == 'test'}
            
        filtered_idxs = []
        dropped = 0
        for idx in self.my_model_sorted_idxs:
            image_path = self.my_model_preds['image_paths'][idx]
            phrase = self.my_model_preds['phrases'][idx]
            lookup_key = (get_filename_without_extension(image_path), phrase)
            if lookup_key in self.maira2_map and lookup_key in self.biovilt_map:
                filtered_idxs.append(idx)
            else:
                dropped += 1
        self.my_model_sorted_idxs = filtered_idxs
        print(f"Filtered indices: kept {len(filtered_idxs)}, dropped {dropped}")

    def _adapt_my_model_prediction(self, index):
        if not self.use_gt_polygons:
            iou_thr, conf_thr, pre_nms_max_det, post_nms_max_det = (
                self.my_model_metrics['best_bbox_iou_th'], self.my_model_metrics['best_bbox_conf_th'],
                self.my_model_metrics['best_bbox_pre_nms_max_det'], self.my_model_metrics['best_bbox_post_nms_max_det']
            )
        else:
            conf_thr = self.my_model_metrics['best_prob_conf_th']
            iou_thr, pre_nms_max_det, post_nms_max_det = 0.5, 1000, 100

        heatmap = self.my_model_preds['pred_bbox_prob_maps'][index]
        pred_bbox_probs = self.my_model_preds['pred_bbox_prob_maps'][index].flatten()
        pred_bbox_coords = self.my_model_preds['pred_bbox_coord_maps'][index].reshape(-1, 4)
        phrase = self.my_model_preds['phrases'][index]
        image_path = self.my_model_preds['image_paths'][index]
        gt_bbox_coords = self.my_model_preds['gt_bbox_coords'][index]
        
        gt_polygons = None
        gt_mask = None
        
        if self.use_gt_polygons:
            gt_polygons = self.my_model_preds['gt_polygons'][index]
            gt_mask = self.my_model_preds['gt_mask'][index]

        pred_bbox_probs = torch.tensor(pred_bbox_probs, dtype=torch.float32)
        pred_bbox_coords = torch.tensor(pred_bbox_coords, dtype=torch.float32)
        gt_bbox_coords = torch.tensor(gt_bbox_coords, dtype=torch.float32)

        if self.my_model_bbox_format == 'cxcywh':
            pred_bbox_coords = cxcywh_to_xyxy_tensor(pred_bbox_coords)
            if gt_bbox_coords.shape[0] > 0:
                gt_bbox_coords = cxcywh_to_xyxy_tensor(gt_bbox_coords)

        mask = pred_bbox_probs > conf_thr
        pred_bbox_coords, pred_bbox_probs = pred_bbox_coords[mask], pred_bbox_probs[mask]

        if len(pred_bbox_probs) > pre_nms_max_det:
            pred_bbox_probs, idxs = torch.topk(pred_bbox_probs, pre_nms_max_det)
            pred_bbox_coords = pred_bbox_coords[idxs]
        if len(pred_bbox_probs) > 0:
            keep = nms(pred_bbox_coords, pred_bbox_probs, iou_thr)
            pred_bbox_coords, pred_bbox_probs = pred_bbox_coords[keep], pred_bbox_probs[keep]
        if len(pred_bbox_probs) > post_nms_max_det:
            pred_bbox_probs, idxs = torch.topk(pred_bbox_probs, post_nms_max_det)
            pred_bbox_coords = pred_bbox_coords[idxs]

        return {'image_path': image_path, 'phrase': phrase, 'pred_bboxes': pred_bbox_coords.numpy(),
                'gt_bboxes': gt_bbox_coords.numpy(), 'heatmap': heatmap,
                'gt_polygons': gt_polygons, 'gt_mask': gt_mask}

    def _visualize_single_prediction(
        self, ax, title, image_path, gt_bboxes, pred_bboxes=None, heatmap=None, 
        gt_polygons=None, fontsize=20
    ):
        try:
            img = Image.open(image_path).convert('RGB')
            ax.imshow(img, cmap='gray'); width, height = img.size
        except FileNotFoundError:
            ax.text(0.5, 0.5, f"Image not found:\n{os.path.basename(image_path)}", ha='center', va='center', wrap=True)
            ax.set_title(title); ax.axis('off'); return

        if heatmap is not None:
            resized_heatmap = cv2.resize(heatmap, (width, height), interpolation=cv2.INTER_CUBIC)
            ax.imshow(resized_heatmap, cmap='viridis', alpha=0.4)

        if gt_bboxes is not None and len(gt_bboxes) > 0:
            if not isinstance(gt_bboxes[0], (list, np.ndarray)): gt_bboxes = np.array([gt_bboxes])
            for bbox in gt_bboxes:
                x1, y1, x2, y2 = bbox
                rect = patches.Rectangle((x1 * width, y1 * height), (x2 - x1) * width, (y2 - y1) * height, linewidth=2, edgecolor='g', facecolor='none')
                ax.add_patch(rect)
        
        if gt_polygons is not None and len(gt_polygons) > 0:
            # UNDO: Flattening is removed. Assumes gt_polygons is a simple list of polygons.
            for poly in gt_polygons:
                if not poly or len(poly) == 0: continue
                if isinstance(poly[0][0], list): poly = poly[0] # Defensive check
                scaled_poly = [[p[0] * width, p[1] * height] for p in poly]
                patch = patches.Polygon(scaled_poly, closed=True, linewidth=2, edgecolor='g', facecolor='none')
                ax.add_patch(patch)

        if pred_bboxes is not None:
            for bbox in pred_bboxes:
                x1, y1, x2, y2 = bbox
                rect = patches.Rectangle((x1 * width, y1 * height), (x2 - x1) * width, (y2 - y1) * height, linewidth=2, edgecolor='r', facecolor='none')
                ax.add_patch(rect)
        ax.set_title(title, fontsize=fontsize); ax.axis('off')

    def _compute_all_metrics(self, gt_bboxes=None, pred_bboxes=None, heatmap=None, 
                             gt_polygons=None, gt_mask=None, mask_size=(100, 100)):
        metrics = {}
        if gt_mask is None:
            if self.use_gt_polygons and gt_polygons is not None and len(gt_polygons) > 0:
                # UNDO: Flattening is removed.
                gt_mask = polygons_to_mask(gt_polygons, mask_size[0], mask_size[1])
            elif gt_bboxes is not None and gt_bboxes.any():
                # print(gt_bboxes)
                gt_mask = convert_bboxes_into_presence_map(gt_bboxes, mask_size)
            else:
                return {k: float('nan') for k in ['iou_bbox', 'iou_seg', 'cnr', 'dice', 'soft_dice', 'precision', 'recall', 'rand_index']}

        if heatmap is not None:
            best_iou, best_th = -1, 0.0
            for th in np.arange(0.1, 1.0, 0.1):
                iou = calculate_segmentation_iou(gt_mask, heatmap, threshold=th)
                if iou > best_iou: best_iou, best_th = iou, th
            metrics.update({'iou_seg': best_iou, 'best_th': best_th, 'cnr': calculate_cnr(gt_mask, heatmap),
                            'soft_dice': calculate_soft_dice(gt_mask, heatmap), 'dice': calculate_dice(gt_mask, heatmap, threshold=best_th),
                            'precision': calculate_segmentation_precision(gt_mask, heatmap, threshold=best_th),
                            'recall': calculate_segmentation_recall(gt_mask, heatmap, threshold=best_th),
                            'rand_index': calculate_rand_index(gt_mask, heatmap, threshold=best_th)})

        if pred_bboxes is not None and gt_bboxes is not None and gt_bboxes.any():
            metrics['iou_bbox'] = compute_bbox_union_iou(gt_bboxes, pred_bboxes)
            if 'dice' not in metrics:
                pred_mask = convert_bboxes_into_presence_map(pred_bboxes, mask_size)
                metrics.update({'cnr': calculate_cnr(gt_mask, pred_mask), 'dice': calculate_dice(gt_mask, pred_mask),
                                'soft_dice': calculate_soft_dice(gt_mask, pred_mask), 'precision': calculate_segmentation_precision(gt_mask, pred_mask),
                                'recall': calculate_segmentation_recall(gt_mask, pred_mask), 'rand_index': calculate_rand_index(gt_mask, pred_mask),
                                'iou_seg': calculate_dice(gt_mask, pred_mask)})
        return metrics

    def visualize_comparison(self, index):
        if not (0 <= index < len(self.my_model_sorted_idxs)):
            print(f"Error: Index {index} is out of bounds."); return
        
        index = self.my_model_sorted_idxs[index]
        my_model_data = self._adapt_my_model_prediction(index)
        image_path, phrase = my_model_data['image_path'], my_model_data['phrase']
        
        # MODIFIED: Use my_model_data as the single source of truth for GT
        gt_bboxes_truth = my_model_data['gt_bboxes']
        if not isinstance(gt_bboxes_truth[0], (list, np.ndarray)): gt_bboxes_truth = np.array([gt_bboxes_truth])
        gt_polygons_truth = None
        if self.use_gt_polygons:
            gt_polygons_truth = my_model_data['gt_polygons']
        
        print(f'image_path = {image_path}')
        lookup_key = (get_filename_without_extension(image_path), phrase)
        print('lookup_key =', lookup_key)

        maira2_data = self.maira2_map.get(lookup_key)
        biovilt_data = self.biovilt_map.get(lookup_key)
        
        print(f'maira2_data: {maira2_data}')

        fig, axes = plt.subplots(1, 3, figsize=(24, 8))
        fig.suptitle(f'Phrase: "{phrase}"\n(GT: Green, Pred: Red)', fontsize=25, y=0.98)

        self._visualize_single_prediction(axes[0], "My Model", image_path, gt_bboxes_truth,
                                          pred_bboxes=my_model_data['pred_bboxes'], heatmap=my_model_data['heatmap'],
                                          gt_polygons=gt_polygons_truth)
        my_model_metrics = self._compute_all_metrics(gt_bboxes=gt_bboxes_truth, pred_bboxes=my_model_data['pred_bboxes'],
                                                     heatmap=my_model_data['heatmap'], gt_mask=my_model_data['gt_mask'])

        maira2_metrics = None
        if maira2_data:
            pred_bboxes = np.array(maira2_data.get('predicted_bboxes_maira2', maira2_data.get('predicted_bboxes', [])))
            # MODIFIED: Use GT from my_model_data
            self._visualize_single_prediction(axes[1], "MAIRA-2", image_path, gt_bboxes_truth, pred_bboxes=pred_bboxes, gt_polygons=gt_polygons_truth)
            maira2_metrics = self._compute_all_metrics(gt_bboxes=gt_bboxes_truth, pred_bboxes=pred_bboxes, gt_polygons=gt_polygons_truth)
        else:
            axes[1].text(0.5, 0.5, "Data not found", ha='center', va='center'); axes[1].set_title("MAIRA-2"); axes[1].axis('off')

        biovilt_metrics = None
        if biovilt_data:
            # MODIFIED: Use GT from my_model_data
            self._visualize_single_prediction(axes[2], "BioViL-T", image_path, gt_bboxes_truth, heatmap=biovilt_data['similarity_map'], gt_polygons=gt_polygons_truth)
            biovilt_metrics = self._compute_all_metrics(gt_bboxes=gt_bboxes_truth, heatmap=biovilt_data['similarity_map'], gt_polygons=gt_polygons_truth)
        else:
            axes[2].text(0.5, 0.5, "Data not found", ha='center', va='center'); axes[2].set_title("BioViL-T"); axes[2].axis('off')

        plt.tight_layout(rect=[0, 0, 1, 0.95]); plt.show()

        def format_metric(metrics, key):
            if metrics is None or np.isnan(metrics.get(key, float('nan'))): return "---"
            val = metrics[key]
            if key == 'iou_seg' and 'best_th' in metrics: return f"{val:.3f} (th={metrics['best_th']:.1f})"
            return f"{val:.3f}"

        metric_keys = ['iou_bbox', 'iou_seg', 'dice', 'soft_dice', 'cnr', 'precision', 'recall', 'rand_index']
        metric_names = ['IoU (BBox)', 'IoU (Seg)', 'Dice', 'Soft Dice', 'CNR', 'Precision', 'Recall', 'Rand Index']
        all_metrics_data = [my_model_metrics, maira2_metrics, biovilt_metrics]
        
        table_html = """
        <style>
            .metrics-table { width: 80%; margin-left: auto; margin-right: auto; border-collapse: collapse; font-family: sans-serif; }
            .metrics-table th, .metrics-table td { border: 1px solid #ddd; padding: 8px; }
            .metrics-table th { padding-top: 12px; padding-bottom: 12px; text-align: center; background-color: #f2f2f2; font-weight: bold; }
            .metrics-table tr:nth-child(even){background-color: #f9f9f9;}
            .metrics-table td:first-child { font-weight: bold; }
            .metrics-table td { text-align: center; }
        </style>
        <table class="metrics-table">
            <tr><th>Metric</th><th>My Model</th><th>MAIRA-2</th><th>BioViL-T</th></tr>
        """
        for name, key in zip(metric_names, metric_keys):
            table_html += f"<tr><td>{name}</td>"
            for metrics_dict in all_metrics_data:
                table_html += f"<td>{format_metric(metrics_dict, key)}</td>"
            table_html += "</tr>"
        table_html += "</table>"
        display(HTML(table_html))

# MS-CXR

In [ ]:
!ls "/mnt/data/pamessina/workspaces/medvqa-workspace/results/phrase_grounding/20250727_125948_mscxr+chst-img-alg+chst-img-pg+vinbig+padchest-gr_PhraseGrounder(microsoft-rad-dino-maira-2,AdaptiveFiLM_MLP_BBoxRegression,128,256,256-128)"

In [ ]:
# --- Example Usage ---

# Define the paths to your data files
MY_MODEL_DIR = "/mnt/data/pamessina/workspaces/medvqa-workspace/results/phrase_grounding/20250727_125948_mscxr+chst-img-alg+chst-img-pg+vinbig+padchest-gr_PhraseGrounder(microsoft-rad-dino-maira-2,AdaptiveFiLM_MLP_BBoxRegression,128,256,256-128)/"
my_model_preds_path = os.path.join(MY_MODEL_DIR, 'mscxr_predictions_and_gt.pkl')
my_model_metrics_path = os.path.join(MY_MODEL_DIR, 'mscxr_predictions_and_gt.pkl.metrics.pkl')

maira2_preds_path = '/home/pamessina/maira2_phrase_grounding_predictions/maira2_mscxr_test_set_predictions.json'
biovilt_preds_path = '/home/pamessina/biovil-t_phrase_grounding_predictions/mscxr_phrase_grounding_results.pkl'

# Create an instance of the comparator
# This will load and process all the data.
msccxr_comparator = PhraseGroundingComparator(
    my_model_preds_path=my_model_preds_path,
    my_model_metrics_path=my_model_metrics_path,
    maira2_preds_path=maira2_preds_path,
    biovilt_preds_path=biovilt_preds_path,
    sort_indices=False,
)

In [ ]:
msccxr_comparator.visualize_comparison(0)

In [ ]:
msccxr_comparator.visualize_comparison(1)

In [ ]:
msccxr_comparator.visualize_comparison(2)

In [ ]:
msccxr_comparator.visualize_comparison(3)

In [ ]:
msccxr_comparator.visualize_comparison(21)

In [ ]:
msccxr_comparator.visualize_comparison(101)

# PadChest-GR

In [ ]:
!ls "/mnt/data/pamessina/workspaces/medvqa-work-space/results/phrase_grounding/20250727_060714_mscxr+chst-img-alg+chst-img-pg+vinbig+padchest-gr_PhraseGrounder(microsoft-rad-dino-maira-2,AdaptiveFiLM_MLP_BBoxRegression,128,256,256-128)"

In [ ]:
# --- Example Usage ---

# Define the paths to your data files
MY_MODEL_DIR = "/mnt/data/pamessina/workspaces/medvqa-workspace/results/phrase_grounding/20250727_060714_mscxr+chst-img-alg+chst-img-pg+vinbig+padchest-gr_PhraseGrounder(microsoft-rad-dino-maira-2,AdaptiveFiLM_MLP_BBoxRegression,128,256,256-128)/"
my_model_preds_path = os.path.join(MY_MODEL_DIR, 'padchestgr_predictions_and_gt.pkl')
my_model_metrics_path = os.path.join(MY_MODEL_DIR, 'padchestgr_predictions_and_gt.pkl.metrics.pkl')

maira2_preds_path = '/home/pamessina/maira2_phrase_grounding_predictions/padchest_gr_phrase_grounding_results.jsonl'
biovilt_preds_path = '/home/pamessina/biovil-t_phrase_grounding_predictions/padchest_gr_phrase_grounding_results.pkl'

# Create an instance of the comparator
# This will load and process all the data.
padchestgr_comparator = PhraseGroundingComparator(
    my_model_preds_path=my_model_preds_path,
    my_model_metrics_path=my_model_metrics_path,
    maira2_preds_path=maira2_preds_path,
    biovilt_preds_path=biovilt_preds_path,
)

In [ ]:
padchestgr_comparator.visualize_comparison(0)

In [ ]:
padchestgr_comparator.visualize_comparison(1)

In [ ]:
padchestgr_comparator.visualize_comparison(10)

In [ ]:
padchestgr_comparator.visualize_comparison(100)

In [ ]:
padchestgr_comparator.visualize_comparison(200)

In [ ]:
padchestgr_comparator.visualize_comparison(250)

In [ ]:
padchestgr_comparator.visualize_comparison(300)

# VinDr-CXR

In [ ]:
!ls "/mnt/data/pamessina/workspaces/medvqa-workspace/results/phrase_grounding/20250727_172941_vinbig_PhraseGrounder(microsoft-rad-dino-maira-2,AdaptiveFiLM_MLP_BBoxRegression,128,256,256-128)"

In [ ]:
# --- Example Usage ---

# Define the paths to your data files
MY_MODEL_DIR = "/mnt/data/pamessina/workspaces/medvqa-workspace/results/phrase_grounding/20250727_172941_vinbig_PhraseGrounder(microsoft-rad-dino-maira-2,AdaptiveFiLM_MLP_BBoxRegression,128,256,256-128)/"
my_model_preds_path = os.path.join(MY_MODEL_DIR, 'vindrcxr_predictions_and_gt.pkl')
my_model_metrics_path = os.path.join(MY_MODEL_DIR, 'vindrcxr_predictions_and_gt.pkl.metrics.pkl')

maira2_preds_path = '/home/pamessina/maira2_phrase_grounding_predictions/vindrcxr_phrase_grounding_results.jsonl'
biovilt_preds_path = '/home/pamessina/biovil-t_phrase_grounding_predictions/vindrcxr_phrase_grounding_results.pkl'

# Create an instance of the comparator
# This will load and process all the data.
vindrcxr_comparator = PhraseGroundingComparator(
    my_model_preds_path=my_model_preds_path,
    my_model_metrics_path=my_model_metrics_path,
    maira2_preds_path=maira2_preds_path,
    biovilt_preds_path=biovilt_preds_path,
    using_vindrcxr=True,
)

In [ ]:
vindrcxr_comparator.visualize_comparison(0)

In [ ]:
vindrcxr_comparator.visualize_comparison(100)

In [ ]:
vindrcxr_comparator.visualize_comparison(500)

In [ ]:
vindrcxr_comparator.visualize_comparison(600)

In [ ]:
vindrcxr_comparator.visualize_comparison(700)

In [ ]:
vindrcxr_comparator.visualize_comparison(800)

# Chest ImaGenome

In [ ]:
!ls "/mnt/data/pamessina/workspaces/medvqa-workspace/results/phrase_grounding/20250817_143800_mscxr+chst-img-alg+chst-img-pg+vinbig+padchest-gr_PhraseGrounder(microsoft-rad-dino-maira-2,AdaptiveFiLM_MLP_BBoxRegression,128,256,256-128)"

In [ ]:
!ls "/home/pamessina/biovil-t_phrase_grounding_predictions/"

In [ ]:
# --- Example Usage ---

# Define the paths to your data files
MY_MODEL_DIR = "/mnt/data/pamessina/workspaces/medvqa-workspace/results/phrase_grounding/20250817_143800_mscxr+chst-img-alg+chst-img-pg+vinbig+padchest-gr_PhraseGrounder(microsoft-rad-dino-maira-2,AdaptiveFiLM_MLP_BBoxRegression,128,256,256-128)/"
my_model_preds_path = os.path.join(MY_MODEL_DIR, 'chestimagenome_predictions_and_gt.pkl')
my_model_metrics_path = os.path.join(MY_MODEL_DIR, 'chestimagenome_predictions_and_gt.pkl.metrics.pkl')

maira2_preds_path = '/home/pamessina/maira2_phrase_grounding_predictions/chestimagenome_phrase_grounding_results.jsonl'
biovilt_preds_path = '/home/pamessina/biovil-t_phrase_grounding_predictions/chest_imagenome_phrase_grounding_results.pkl'

# Create an instance of the comparator
# This will load and process all the data.
chestimagenome_comparator = PhraseGroundingComparator(
    my_model_preds_path=my_model_preds_path,
    my_model_metrics_path=my_model_metrics_path,
    maira2_preds_path=maira2_preds_path,
    biovilt_preds_path=biovilt_preds_path,
)

In [ ]:
chestimagenome_comparator.visualize_comparison(0)

In [ ]:
chestimagenome_comparator.visualize_comparison(1)

In [ ]:
chestimagenome_comparator.visualize_comparison(2)

In [ ]:
chestimagenome_comparator.visualize_comparison(9)

# ChestX-Det

In [ ]:
!ls "/mnt/data/pamessina/workspaces/medvqa-workspace/results/phrase_grounding/20250727_172941_vinbig_PhraseGrounder(microsoft-rad-dino-maira-2,AdaptiveFiLM_MLP_BBoxRegression,128,256,256-128)"

In [ ]:
# --- Example Usage for chestX-det ---

# Define the paths to your data files
MY_MODEL_DIR = "/mnt/data/pamessina/workspaces/medvqa-workspace/results/phrase_grounding/20250727_172941_vinbig_PhraseGrounder(microsoft-rad-dino-maira-2,AdaptiveFiLM_MLP_BBoxRegression,128,256,256-128)/"
my_model_preds_path = os.path.join(MY_MODEL_DIR, 'chestxdet_predictions_and_gt.pkl')
my_model_metrics_path = os.path.join(MY_MODEL_DIR, 'chestxdet_predictions_and_gt.pkl.metrics.pkl')

maira2_preds_path = '/home/pamessina/maira2_phrase_grounding_predictions/chestxdet_phrase_grounding_results.jsonl'
biovilt_preds_path = '/home/pamessina/biovil-t_phrase_grounding_predictions/chestxdet_phrase_grounding_results.pkl'

# Create an instance of the comparator with the new flag
chestxdet_comparator = PhraseGroundingComparator(
    my_model_preds_path=my_model_preds_path,
    my_model_metrics_path=my_model_metrics_path,
    maira2_preds_path=maira2_preds_path,
    biovilt_preds_path=biovilt_preds_path,
    use_gt_polygons=True
)

In [ ]:
chestxdet_comparator.visualize_comparison(0)

In [ ]:
chestxdet_comparator.visualize_comparison(1)

In [ ]:
chestxdet_comparator.visualize_comparison(2)

In [ ]:
chestxdet_comparator.visualize_comparison(10)

In [ ]:
chestxdet_comparator.visualize_comparison(30)

In [ ]:
chestxdet_comparator.visualize_comparison(100)

In [ ]:
chestxdet_comparator.visualize_comparison(500)

In [ ]:
chestxdet_comparator.visualize_comparison(600)

In [ ]:
chestxdet_comparator.visualize_comparison(700)

In [ ]:
chestxdet_comparator.visualize_comparison(1000)

In [ ]:
chestxdet_comparator.visualize_comparison(1001)

In [ ]:
chestxdet_comparator.visualize_comparison(1002)

In [ ]:
chestxdet_comparator.visualize_comparison(1050)

# Metric plots for MS-CXR

In [ ]:
from importlib import reload
import medvqa

In [ ]:
reload(medvqa.evaluation.plots)

In [ ]:
import random
import numpy as np
import pandas as pd
from tqdm import tqdm
from abc import ABC, abstractmethod

from medvqa.datasets.ms_cxr import get_ms_cxr_phrase_to_category_name
from medvqa.evaluation.bootstrapping import apply_bootstrapping
from medvqa.evaluation.plots import plot_metric_bars_per_method
from medvqa.metrics.bbox.utils import find_optimal_probability_map_conf_threshold
from medvqa.utils.bbox_utils import convert_bboxes_into_presence_map
from medvqa.utils.files_utils import load_json, load_jsonl, load_pickle
from medvqa.utils.metrics_utils import (
    calculate_cnr, calculate_dice, calculate_rand_index, calculate_segmentation_iou,
    calculate_segmentation_precision, calculate_segmentation_recall, calculate_soft_dice
)

# --- 1. Base Class with Shared Logic ---

class BasePhraseGroundingComparator(ABC):
    """
    An abstract base class to load, process, and compare phrase grounding results
    from different models on a given dataset.
    """
    def __init__(self, my_model_metrics_paths, my_model_aliases, 
                 maira2_preds_path, biovilt_preds_path, biovilt_threshold=None):
        """
        Initializes the comparator by loading and processing data for all models.

        Args:
            my_model_metrics_paths (list of str): A list of file paths to the 
                                                  pre-computed metrics for your models.
            my_model_aliases (list of str): A list of display names for your models.
            maira2_preds_path (str): Path to MAIRA-2 prediction file.
            biovilt_preds_path (str): Path to BioViL-T prediction file.
            biovilt_threshold (float, optional): A pre-computed optimal confidence
                                                 threshold for BioViL-T. If None,
                                                 it will be computed automatically.
        """
        print(f"Initializing {self.__class__.__name__}...")
        assert len(my_model_metrics_paths) == len(my_model_aliases), \
            "The number of model paths must match the number of model aliases."
        
        self.my_model_metrics_paths = my_model_metrics_paths
        self.my_model_aliases = my_model_aliases
        self.maira2_preds_path = maira2_preds_path
        self.biovilt_preds_path = biovilt_preds_path
        self.biovilt_threshold = biovilt_threshold

        # Load and process data for each model
        if self.my_model_metrics_paths:
            self._load_my_models()
        else:
            self.my_models_metrics_list = []
            self.my_models_method_dicts_list = []
        
        self._load_and_process_maira2()
        self._load_and_process_biovilt()
        print("Initialization complete.")

    def _load_my_models(self):
        """Loads pre-computed metrics for all provided 'My Model' experiments."""
        print(f"Loading metrics for {len(self.my_model_metrics_paths)} custom models...")
        self.my_models_metrics_list = []
        self.my_models_method_dicts_list = []

        for path in tqdm(self.my_model_metrics_paths, desc="Loading custom models"):
            metrics = load_pickle(path)['test_metrics']
            self.my_models_metrics_list.append(metrics)
            
            method_dicts_for_model = {}
            for metric_key, metric_data in metrics.items():
                if isinstance(metric_data, dict) and metric_data and 'mean' in list(metric_data.values())[0]:
                    current_metric_dict = {}
                    for name, values in metric_data.items():
                        current_metric_dict[name] = values['mean']
                        current_metric_dict[f'{name}_std'] = values['std']
                    method_dicts_for_model[metric_key] = current_metric_dict
            self.my_models_method_dicts_list.append(method_dicts_for_model)
        print("Custom models' metrics loaded.")

    @abstractmethod
    def _load_and_process_maira2(self):
        """
        Abstract method to load MAIRA-2 predictions and compute metrics.
        Must be implemented by subclasses.
        """
        pass

    @abstractmethod
    def _load_and_process_biovilt(self):
        """
        Abstract method to load BioViL-T predictions and compute metrics.
        Must be implemented by subclasses.
        """
        pass

    def plot_comparison(self, metric_key, title=None, custom_alias_map=None, **kwargs):
        """
        Generates a bar plot comparing the models on a given metric.
        """
        if not self.my_models_metrics_list:
             raise ValueError("Cannot plot comparison without at least one custom model to define metric keys.")
        
        if metric_key not in self.my_models_metrics_list[0]:
            raise ValueError(f"Metric key '{metric_key}' not found in the first model's metrics.")

        default_alias_map = {'iou': 'IoU', 'cnr': 'CNR', 'dice': 'Dice', 'soft dice': 'Soft Dice', 'rand index': 'Rand Index', 'precision': 'Precision', 'recall': 'Recall'}
        final_alias_map = default_alias_map.copy()
        if custom_alias_map:
            final_alias_map.update(custom_alias_map)

        metric_names = list(self.my_models_metrics_list[0][metric_key].keys())
        overall_metric = [m for m in metric_names if '(' not in m]
        class_metrics = [m for m in metric_names if '(' in m]
        class_metrics.sort()
        metric_names = class_metrics + overall_metric
        
        metric_aliases = []
        for name in metric_names:
            clean_name = name.replace('prob_', '').replace('bbox_', '').replace('_', ' ')
            if '(' in clean_name:
                parts = clean_name.split('(', 1)
                metric_part = parts[0].strip()
                class_part = f"({parts[1]}"
                display_metric = final_alias_map.get(metric_part, metric_part.capitalize())
                alias = f"{display_metric}{class_part}"
                if hasattr(self, '_category_to_count'):
                    class_name = class_part[1:-1].strip()
                    count = self._category_to_count[class_name]
                    alias += f" [{count}]"
            else:
                metric_part = clean_name.strip()
                alias = final_alias_map.get(metric_part, metric_part.capitalize())
                if hasattr(self, '_total_count'):
                    alias += f" [{self._total_count}]"
            metric_aliases.append(alias)

        method_dicts = []
        for model_dicts in self.my_models_method_dicts_list:
            method_dicts.append(model_dicts[metric_key])
        
        method_dicts.extend([self.maira2_method_dict, self.biovilt_method_dict])

        method_aliases = self.my_model_aliases[:]
        method_aliases.extend(['MAIRA-2', 'BioViL-T'])

        plot_settings = {'title': title or f'Phrase Grounding Comparison: {metric_key.replace("_with_bootstrapping", "")}', 'vertical': True, 'show_std': True, 'xtick_rotation': 45, 'figsize': (12, 7), 'ylabel': metric_key.split('_')[1].upper() if '_' in metric_key else 'Score', 'ylim': [0, 1.05], 'bbox_to_anchor': (0.5, -0.25)}
        plot_settings.update(kwargs)

        plot_metric_bars_per_method(
            method_dicts=method_dicts,
            method_aliases=method_aliases,
            metric_names=metric_names,
            metric_aliases=metric_aliases,
            **plot_settings
        )

# --- 2. Original Class, now inheriting from the Base Class ---

class MSCXR_PhraseGroundingComparator(BasePhraseGroundingComparator):
    """
    Comparator for phrase grounding results on the MS-CXR dataset.
    """
    def __init__(self, **kwargs):
        super().__init__(**kwargs)

    def _load_and_process_maira2(self):
        print("Loading and processing MAIRA-2 predictions for MS-CXR...")
        maira2_predictions = load_json(self.maira2_preds_path)
        categories_list = [x['category_name'] for x in maira2_predictions]
        category_names = sorted(list(set(categories_list)))
        category_to_idx = {name: i for i, name in enumerate(category_names)}
        class_to_indices = [[] for _ in category_names]
        for i, category in enumerate(categories_list):
            class_to_indices[category_to_idx[category]].append(i)
        
        self.maira2_metrics = {}
        ious_list = [x['iou'] for x in maira2_predictions]
        cnrs_list, dice_list, soft_dice_list, precision_list, recall_list, rand_index_list = [], [], [], [], [], []
        
        for x in tqdm(maira2_predictions, desc="Calculating MAIRA-2 metrics"):
            gt_mask = convert_bboxes_into_presence_map(x['gt_bboxes'], (100, 100))
            pred_mask = convert_bboxes_into_presence_map(x['predicted_bboxes_maira2'], (100, 100))
            cnrs_list.append(calculate_cnr(gt_mask, pred_mask))
            dice_list.append(calculate_dice(gt_mask, pred_mask))
            soft_dice_list.append(calculate_soft_dice(gt_mask, pred_mask))
            precision_list.append(calculate_segmentation_precision(gt_mask, pred_mask))
            recall_list.append(calculate_segmentation_recall(gt_mask, pred_mask))
            rand_index_list.append(calculate_rand_index(gt_mask, pred_mask))
        
        metric_lists = {'cnr': cnrs_list, 'iou': ious_list, 'dice': dice_list, 'soft_dice': soft_dice_list, 'precision': precision_list, 'recall': recall_list, 'rand_index': rand_index_list}
        for name, values in metric_lists.items():
            self.maira2_metrics[f'{name}_list'] = values
            out = apply_bootstrapping(metric_values=values, class_to_indices=class_to_indices, class_names=category_names, metric_name=name, use_tqdm=False)
            self.maira2_metrics.update(out)
        
        self.maira2_method_dict = {k: v['mean'] for k, v in self.maira2_metrics.items() if isinstance(v, dict)}
        self.maira2_method_dict.update({f"{k}_std": v['std'] for k, v in self.maira2_metrics.items() if isinstance(v, dict)})
        print("MAIRA-2 processing complete.")

    def _load_and_process_biovilt(self):
        print("Loading and processing BioViL-T predictions for MS-CXR...")
        biovilt_predictions = load_pickle(self.biovilt_preds_path)
        phrase_to_category_name = get_ms_cxr_phrase_to_category_name()
        
        test_predictions = [x for x in biovilt_predictions if x['split'] == 'test']
        categories_list = [phrase_to_category_name[x['phrase']] for x in test_predictions]
        category_names = sorted(list(set(categories_list)))
        category_to_idx = {name: i for i, name in enumerate(category_names)}
        class_to_indices = [[] for _ in category_names]
        for i, category in enumerate(categories_list):
            class_to_indices[category_to_idx[category]].append(i)
            
        self._category_to_count = { category: len(class_to_indices[category_to_idx[category]]) for category in category_names }
        self._total_count = len(categories_list)
        
        pred_mask_list = [x['similarity_map'] for x in test_predictions]
        gt_bboxes_list = [x['gt_bboxes'] for x in test_predictions]

        if self.biovilt_threshold is None:
            print("Finding optimal threshold for BioViL-T...")
            sample_indices = random.sample(range(len(pred_mask_list)), min(50, len(pred_mask_list)))
            pred_mask_sample = [pred_mask_list[i] for i in sample_indices]
            gt_bboxes_sample = [gt_bboxes_list[i] for i in sample_indices]
            out = find_optimal_probability_map_conf_threshold(np.array(pred_mask_sample), gt_bboxes_sample)
            best_conf_th = out['best_conf_th']
        else:
            print(f"Using provided BioViL-T threshold: {self.biovilt_threshold}")
            best_conf_th = self.biovilt_threshold
        
        print(f"BioViL-T optimal threshold: {best_conf_th}")
        
        self.biovilt_metrics = {}
        ious_list, cnrs_list, dice_list, soft_dice_list, precision_list, recall_list, rand_index_list = [], [], [], [], [], [], []
        for pred_mask, gt_bboxes in tqdm(zip(pred_mask_list, gt_bboxes_list), total=len(pred_mask_list), desc="Calculating BioViL-T metrics"):
            gt_mask = convert_bboxes_into_presence_map(gt_bboxes, (100, 100))
            ious_list.append(calculate_segmentation_iou(gt_mask, pred_mask, threshold=best_conf_th))
            cnrs_list.append(calculate_cnr(gt_mask, pred_mask))
            dice_list.append(calculate_dice(gt_mask, pred_mask, threshold=best_conf_th))
            soft_dice_list.append(calculate_soft_dice(gt_mask, pred_mask))
            precision_list.append(calculate_segmentation_precision(gt_mask, pred_mask, threshold=best_conf_th))
            recall_list.append(calculate_segmentation_recall(gt_mask, pred_mask, threshold=best_conf_th))
            rand_index_list.append(calculate_rand_index(gt_mask, pred_mask, threshold=best_conf_th))
        
        metric_lists = {'cnr': cnrs_list, 'iou': ious_list, 'dice': dice_list, 'soft_dice': soft_dice_list, 'precision': precision_list, 'recall': recall_list, 'rand_index': rand_index_list}
        for name, values in metric_lists.items():
            self.biovilt_metrics[f'{name}_list'] = values
            out = apply_bootstrapping(metric_values=values, class_to_indices=class_to_indices, class_names=category_names, metric_name=name, use_tqdm=False)
            self.biovilt_metrics.update(out)
        
        self.biovilt_method_dict = {k: v['mean'] for k, v in self.biovilt_metrics.items() if isinstance(v, dict)}
        self.biovilt_method_dict.update({f"{k}_std": v['std'] for k, v in self.biovilt_metrics.items() if isinstance(v, dict)})
        print("BioViL-T processing complete.")

In [ ]:
# Define paths and aliases for own models
my_model_metrics_paths = [
    "/mnt/data/pamessina/workspaces/medvqa-workspace/results/phrase_grounding/20250727_125948_mscxr+chst-img-alg+chst-img-pg+vinbig+padchest-gr_PhraseGrounder(microsoft-rad-dino-maira-2,AdaptiveFiLM_MLP_BBoxRegression,128,256,256-128)/mscxr_predictions_and_gt.pkl.metrics.pkl",
    "/mnt/data/pamessina/workspaces/medvqa-workspace/results/phrase_grounding/20250727_060714_mscxr+chst-img-alg+chst-img-pg+vinbig+padchest-gr_PhraseGrounder(microsoft-rad-dino-maira-2,AdaptiveFiLM_MLP_BBoxRegression,128,256,256-128)/mscxr_predictions_and_gt.pkl.metrics.pkl",
    "/mnt/data/pamessina/workspaces/medvqa-workspace/results/phrase_grounding/20250723_163744_mscxr_PhraseGrounder(aehrc-cxrmate-rrg24-uniformer,AdaptiveFiLM_MLP_BBoxRegression,128,256,256-128)/mscxr_predictions_and_gt.pkl.metrics.pkl",
]
my_model_aliases = [
    "FG-VLM (RAD-DINO-MAIRA-2; 384x384; tr:MSCXR+PGR+Vin+CIG; val:MSCXR(CNR))",
    "FG-VLM (RAD-DINO-MAIRA-2; 384x384; tr:MSCXR+PGR+Vin+CIG; val:MSCXR+PGR+Vin(CNR))",
    "FG-VLM (UniFormer; 384x384; tr:MSCXR; val:MSCXR(CNR))",
]

# Paths for baselines
maira2_preds_path = '/home/pamessina/maira2_phrase_grounding_predictions/maira2_mscxr_test_set_predictions.json'
biovilt_preds_path = '/home/pamessina/biovil-t_phrase_grounding_predictions/mscxr_phrase_grounding_results.pkl'

# Instantiate the comparator with the lists and a pre-computed threshold
mscxr_comparator = MSCXR_PhraseGroundingComparator(
    my_model_metrics_paths=my_model_metrics_paths,
    my_model_aliases=my_model_aliases,
    maira2_preds_path=maira2_preds_path,
    biovilt_preds_path=biovilt_preds_path,
    biovilt_threshold=0.6, # Pass the known threshold
)

In [ ]:
mscxr_comparator.plot_comparison(
    metric_key='prob_iou_with_bootstrapping',
    title='MS-CXR Phrase Grounding: Intersection over Union (IoU) based on probability maps',
    ylabel='IoU Score',
    sort_methods=True,
    sort_metrics=False,
    bbox_to_anchor=(0.605, 0.77),
    ylim=[0, .9],
    figsize=(12, 6),
    legend_fontsize=9.5,
)

In [ ]:
mscxr_comparator.plot_comparison(
    metric_key='bbox_iou_with_bootstrapping',
    title='MS-CXR Phrase Grounding: Intersection over Union (IoU) based on bouningd boxes',
    ylabel='IoU Score',
    sort_methods=True,
    sort_metrics=False,
    bbox_to_anchor=(0.61, 0.77),
    ylim=[0, 0.82],
    figsize=(12, 6),
    legend_fontsize=9.3,
)

In [ ]:
mscxr_comparator.plot_comparison(
    metric_key='dice_with_bootstrapping',
    title='MS-CXR Phrase Grounding: Dice',
    ylabel='Dice Score',
    sort_methods=True,
    sort_metrics=False,
    bbox_to_anchor=(0.61, 0.77),
    ylim=[0, 1.0],
    figsize=(12, 6),
    legend_fontsize=9.3,
)

In [ ]:
mscxr_comparator.plot_comparison(
    metric_key='cnr_with_bootstrapping',
    title='MS-CXR Phrase Grounding: Contrast-to-Noise Ratio (CNR)',
    ylabel='CNR',
    sort_methods=True,
    sort_metrics=False,
    bbox_to_anchor=(0.61, 0.77),
    ylim=[0, 4.5],
    figsize=(12, 6),
    legend_fontsize=9.3,
)

# Metric plots for PadChest-GR

In [ ]:
import os
import random
import numpy as np
import pandas as pd
from tqdm import tqdm
from medvqa.datasets.padchest import PADCHEST_GR_MASTER_TABLE_CSV_PATH


class PadChestGR_PhraseGroundingComparator(BasePhraseGroundingComparator):
    """
    Comparator for phrase grounding results on the PadChest-GR dataset.
    This class overrides _load_my_models to re-compute bootstrapping
    using label_group for custom models. It also includes fixes for
    key normalization and tracks lookup failures.
    """
    def __init__(self, **kwargs):
        self.master_csv_path = PADCHEST_GR_MASTER_TABLE_CSV_PATH
        self._prepare_category_mapper()
        
        # 3) Initialize a counter for lookup failures
        self.lookup_failures = 0
        
        super().__init__(**kwargs)
        
        # 3) Report the total number of failures after initialization
        print(f"\nTotal category lookup failures: {self.lookup_failures}")
        if self.lookup_failures > 0:
            print("Warning: Some samples could not be mapped to a category.")

    def _prepare_category_mapper(self):
        print("Preparing PadChest-GR category mapper...")
        df = pd.read_csv(self.master_csv_path)
        df_ = df[['ImageID', 'sentence_en', 'label_group']].dropna()
        self.image_id_sentence_to_label_group = {}
        for _, image_id, sentence, label_group in df_.itertuples():
            if sentence.endswith('.'):
                sentence = sentence[:-1]
            self.image_id_sentence_to_label_group[(image_id, sentence)] = label_group

    def _load_my_models(self):
        print(f"Loading metrics for {len(self.my_model_metrics_paths)} custom models and re-computing bootstrapping...")
        self.my_models_metrics_list = []
        self.my_models_method_dicts_list = []

        for path in tqdm(self.my_model_metrics_paths, desc="Processing custom models"):
            metrics_data = load_pickle(path)
            metrics = metrics_data['test_metrics']
            preds_path = path.replace('.metrics.pkl', '')
            preds_data = load_pickle(preds_path)['test_preds_and_gt']
            
            image_paths = preds_data['image_paths']
            phrases = preds_data['phrases']
            
            categories_list = []
            for img_path, phrase in zip(image_paths, phrases):
                image_id = os.path.basename(img_path).replace('.jpg', '.png')
                # Normalize phrase
                if phrase.endswith('.'):
                    phrase = phrase[:-1]
                label_group = self.image_id_sentence_to_label_group.get((image_id, phrase))
                if label_group is None: # 3) Count failures
                    self.lookup_failures += 1
                categories_list.append(label_group)

            category_names = sorted(list(set(c for c in categories_list if c is not None)))
            category_to_idx = {name: i for i, name in enumerate(category_names)}
            class_to_indices = [[] for _ in category_names]
            for i, category in enumerate(categories_list):
                if category is not None:
                    class_to_indices[category_to_idx[category]].append(i)

            metric_map = {
                'prob_ious': 'prob_iou', 'bbox_ious': 'bbox_iou', 'cnrs': 'cnr',
                'soft_dices': 'soft_dice', 'dices': 'dice', 'rand_indices': 'rand_index',
                'precisions': 'precision', 'recalls': 'recall',
            }

            for list_key, metric_name in metric_map.items():
                if list_key in metrics:
                    bootstrapped_key = f'{metric_name}_with_bootstrapping'
                    if metric_name.startswith('prob_iou') or metric_name.startswith('bbox_iou'):
                        metric_name = metric_name[5:]
                    bootstrapped_results = apply_bootstrapping(
                        metric_values=metrics[list_key], class_to_indices=class_to_indices,
                        class_names=category_names, metric_name=metric_name, use_tqdm=False
                    )
                    metrics[bootstrapped_key] = bootstrapped_results

            self.my_models_metrics_list.append(metrics)
            method_dicts_for_model = {}
            for metric_key, metric_data in metrics.items():
                if isinstance(metric_data, dict) and metric_data and 'mean' in list(metric_data.values())[0]:
                    current_metric_dict = {name: v['mean'] for name, v in metric_data.items()}
                    current_metric_dict.update({f"{name}_std": v['std'] for name, v in metric_data.items()})
                    method_dicts_for_model[metric_key] = current_metric_dict
            self.my_models_method_dicts_list.append(method_dicts_for_model)
            
        print("Custom models' metrics loaded and bootstrapping re-computed.")

    def _load_and_process_maira2(self):
        print("Loading and processing MAIRA-2 predictions for PadChest-GR...")
        maira2_predictions = load_jsonl(self.maira2_preds_path)
        
        # --- FIX: Normalize phrase before lookup ---
        categories_list = []
        for x in maira2_predictions:
            phrase = x['phrase']
            if phrase.endswith('.'):
                phrase = phrase[:-1]
            label_group = self.image_id_sentence_to_label_group.get((x['image_id'], phrase))
            if label_group is None: # 3) Count failures
                self.lookup_failures += 1
            categories_list.append(label_group)
        
        # The rest of the method remains the same...
        category_names = sorted(list(set(c for c in categories_list if c is not None)))
        # ... (code continues as before)
        # (The rest of the MAIRA-2 processing logic is unchanged)
        category_to_idx = {name: i for i, name in enumerate(category_names)}
        class_to_indices = [[] for _ in category_names]
        for i, category in enumerate(categories_list):
            if category is not None:
                class_to_indices[category_to_idx[category]].append(i)
        
        self.maira2_metrics = {}
        ious_list, cnrs_list, dice_list, soft_dice_list, precision_list, recall_list, rand_index_list = [], [], [], [], [], [], []
        
        for x in tqdm(maira2_predictions, desc="Calculating MAIRA-2 metrics"):
            gt_mask = convert_bboxes_into_presence_map(x['gt_bboxes'], (200, 200))
            pred_mask = convert_bboxes_into_presence_map(x['predicted_bboxes'], (200, 200))
            ious_list.append(calculate_segmentation_iou(gt_mask, pred_mask))
            cnrs_list.append(calculate_cnr(gt_mask, pred_mask))
            dice_list.append(calculate_dice(gt_mask, pred_mask))
            soft_dice_list.append(calculate_soft_dice(gt_mask, pred_mask))
            precision_list.append(calculate_segmentation_precision(gt_mask, pred_mask))
            recall_list.append(calculate_segmentation_recall(gt_mask, pred_mask))
            rand_index_list.append(calculate_rand_index(gt_mask, pred_mask))
        
        metric_lists = {'cnr': cnrs_list, 'iou': ious_list, 'dice': dice_list, 'soft_dice': soft_dice_list, 'precision': precision_list, 'recall': recall_list, 'rand_index': rand_index_list}
        for name, values in metric_lists.items():
            self.maira2_metrics[f'{name}_list'] = values
            out = apply_bootstrapping(metric_values=values, class_to_indices=class_to_indices, class_names=category_names, metric_name=name, use_tqdm=False)
            self.maira2_metrics.update(out)
        
        self.maira2_method_dict = {k: v['mean'] for k, v in self.maira2_metrics.items() if isinstance(v, dict)}
        self.maira2_method_dict.update({f"{k}_std": v['std'] for k, v in self.maira2_metrics.items() if isinstance(v, dict)})
        print("MAIRA-2 processing complete.")


    def _load_and_process_biovilt(self):
        print("Loading and processing BioViL-T predictions for PadChest-GR...")
        biovilt_predictions = load_pickle(self.biovilt_preds_path)
        test_predictions = [x for x in biovilt_predictions if x['split'] == 'test']
        
        # --- FIX: Normalize phrase before lookup ---
        categories_list = []
        for x in test_predictions:
            phrase = x['phrase']
            if phrase.endswith('.'):
                phrase = phrase[:-1]
            label_group = self.image_id_sentence_to_label_group.get((x['image_id'], phrase))
            if label_group is None: # 3) Count failures
                self.lookup_failures += 1
            categories_list.append(label_group)            

        # The rest of the method remains the same...
        category_names = sorted(list(set(c for c in categories_list if c is not None)))
        # ... (code continues as before)
        # (The rest of the BioViL-T processing logic is unchanged)
        category_to_idx = {name: i for i, name in enumerate(category_names)}
        class_to_indices = [[] for _ in category_names]
        for i, category in enumerate(categories_list):
            if category is not None:
                class_to_indices[category_to_idx[category]].append(i)
                
        self._category_to_count = { category: len(class_to_indices[category_to_idx[category]]) for category in category_names }
        self._total_count = len(categories_list)
        
        pred_mask_list = [x['similarity_map'] for x in test_predictions]
        gt_bboxes_list = [x['gt_bboxes'] for x in test_predictions]

        if self.biovilt_threshold is None:
            print("Finding optimal threshold for BioViL-T...")
            sample_indices = random.sample(range(len(pred_mask_list)), min(50, len(pred_mask_list)))
            pred_mask_sample = [pred_mask_list[i] for i in sample_indices]
            gt_bboxes_sample = [gt_bboxes_list[i] for i in sample_indices]
            out = find_optimal_probability_map_conf_threshold(np.array(pred_mask_sample), gt_bboxes_sample)
            best_conf_th = out['best_conf_th']
        else:
            print(f"Using provided BioViL-T threshold: {self.biovilt_threshold}")
            best_conf_th = self.biovilt_threshold
        
        print(f"BioViL-T optimal threshold: {best_conf_th}")
        
        self.biovilt_metrics = {}
        ious_list, cnrs_list, dice_list, soft_dice_list, precision_list, recall_list, rand_index_list = [], [], [], [], [], [], []
        for pred_mask, gt_bboxes in tqdm(zip(pred_mask_list, gt_bboxes_list), total=len(pred_mask_list), desc="Calculating BioViL-T metrics"):
            gt_mask = convert_bboxes_into_presence_map(gt_bboxes, (100, 100))
            ious_list.append(calculate_segmentation_iou(gt_mask, pred_mask, threshold=best_conf_th))
            cnrs_list.append(calculate_cnr(gt_mask, pred_mask))
            dice_list.append(calculate_dice(gt_mask, pred_mask, threshold=best_conf_th))
            soft_dice_list.append(calculate_soft_dice(gt_mask, pred_mask))
            precision_list.append(calculate_segmentation_precision(gt_mask, pred_mask, threshold=best_conf_th))
            recall_list.append(calculate_segmentation_recall(gt_mask, pred_mask, threshold=best_conf_th))
            rand_index_list.append(calculate_rand_index(gt_mask, pred_mask, threshold=best_conf_th))
        
        metric_lists = {'cnr': cnrs_list, 'iou': ious_list, 'dice': dice_list, 'soft_dice': soft_dice_list, 'precision': precision_list, 'recall': recall_list, 'rand_index': rand_index_list}
        for name, values in metric_lists.items():
            self.biovilt_metrics[f'{name}_list'] = values
            out = apply_bootstrapping(metric_values=values, class_to_indices=class_to_indices, class_names=category_names, metric_name=name, use_tqdm=False)
            self.biovilt_metrics.update(out)
        
        self.biovilt_method_dict = {k: v['mean'] for k, v in self.biovilt_metrics.items() if isinstance(v, dict)}
        self.biovilt_method_dict.update({f"{k}_std": v['std'] for k, v in self.biovilt_metrics.items() if isinstance(v, dict)})
        print("BioViL-T processing complete.")

In [ ]:
# Define paths and aliases for own models
my_model_metrics_paths = [
    "/mnt/data/pamessina/workspaces/medvqa-workspace/results/phrase_grounding/20250817_143800_mscxr+chst-img-alg+chst-img-pg+vinbig+padchest-gr_PhraseGrounder(microsoft-rad-dino-maira-2,AdaptiveFiLM_MLP_BBoxRegression,128,256,256-128)/padchestgr_predictions_and_gt.pkl.metrics.pkl",
    "/mnt/data/pamessina/workspaces/medvqa-workspace/results/phrase_grounding/20250727_060714_mscxr+chst-img-alg+chst-img-pg+vinbig+padchest-gr_PhraseGrounder(microsoft-rad-dino-maira-2,AdaptiveFiLM_MLP_BBoxRegression,128,256,256-128)/padchestgr_predictions_and_gt.pkl.metrics.pkl",
    "/mnt/data/pamessina/workspaces/medvqa-workspace/results/phrase_grounding/20250724_031812_padchest-gr_PhraseGrounder(aehrc-cxrmate-rrg24-uniformer,AdaptiveFiLM_MLP_BBoxRegression,128,256,256-128)/padchestgr_predictions_and_gt.pkl.metrics.pkl",
]
my_model_aliases = [
    "FG-VLM (RAD-DINO-MAIRA-2; 448x448; tr:MSCXR+PGR+Vin+CIG; val:MSCXR+PGR+Vin(SD))",
    "FG-VLM (RAD-DINO-MAIRA-2; 384x384; tr:MSCXR+PGR+Vin+CIG; val:MSCXR+PGR+Vin(CNR))",
    "FG-VLM (UniFormer; 384x384; tr:PGR; val:PGR(CNR))",
]

# Paths for baselines
maira2_preds_path = '/home/pamessina/maira2_phrase_grounding_predictions/padchest_gr_phrase_grounding_results.jsonl'
biovilt_preds_path = '/home/pamessina/biovil-t_phrase_grounding_predictions/padchest_gr_phrase_grounding_results.pkl'

# Instantiate the comparator with the lists and a pre-computed threshold
padchest_gr_comparator = PadChestGR_PhraseGroundingComparator(
    my_model_metrics_paths=my_model_metrics_paths,
    my_model_aliases=my_model_aliases,
    maira2_preds_path=maira2_preds_path,
    biovilt_preds_path=biovilt_preds_path,
    biovilt_threshold=0.6, # Pass the known threshold
)

In [ ]:
padchest_gr_comparator.plot_comparison(
    metric_key='prob_iou_with_bootstrapping',
    title='MS-CXR Phrase Grounding: Intersection over Union (IoU) based on probability maps',
    ylabel='IoU Score',
    sort_methods=True,
    sort_metrics=False,
    bbox_to_anchor=(0.605, 0.77),
    ylim=[0, 1.02],
    figsize=(12, 6),
    legend_fontsize=9.3,
)

In [ ]:
padchest_gr_comparator.plot_comparison(
    metric_key='dice_with_bootstrapping',
    title='MS-CXR Phrase Grounding: Dice',
    ylabel='Dice',
    sort_methods=True,
    sort_metrics=False,
    bbox_to_anchor=(0.605, 0.77),
    ylim=[0, 1.2],
    figsize=(12, 6),
    legend_fontsize=9.5,
)

In [ ]:
padchest_gr_comparator.plot_comparison(
    metric_key='cnr_with_bootstrapping',
    title='MS-CXR Phrase Grounding: CNR',
    ylabel='CNR',
    sort_methods=True,
    sort_metrics=False,
    bbox_to_anchor=(0.605, 0.77),
    ylim=[0, 6.5],
    figsize=(12, 6),
    legend_fontsize=9.5,
)

# Metric plots for VinDr-CXR

In [ ]:
import random
import numpy as np
from tqdm import tqdm

from medvqa.evaluation.bootstrapping import apply_bootstrapping
from medvqa.metrics.bbox.utils import find_optimal_probability_map_conf_threshold
from medvqa.utils.bbox_utils import convert_bboxes_into_presence_map
from medvqa.utils.files_utils import load_jsonl, load_pickle
from medvqa.utils.metrics_utils import (
    calculate_cnr, calculate_dice, calculate_rand_index, calculate_segmentation_iou,
    calculate_segmentation_precision, calculate_segmentation_recall, calculate_soft_dice
)

# --- New Class for VinDr-CXR ---

class VinDrCXR_PhraseGroundingComparator(BasePhraseGroundingComparator):
    """
    Comparator for phrase grounding results on the VinDr-CXR dataset.
    """
    def __init__(self, **kwargs):
        super().__init__(**kwargs)

    def _load_and_process_maira2(self):
        """
        Loads MAIRA-2 predictions for VinDr-CXR and computes metrics.
        """
        print("Loading and processing MAIRA-2 predictions for VinDr-CXR...")
        maira2_predictions = load_jsonl(self.maira2_preds_path)
        
        # In VinDr-CXR, the phrase itself is the category
        categories_list = [x['phrase'] for x in maira2_predictions]
        category_names = sorted(list(set(categories_list)))
        category_to_idx = {name: i for i, name in enumerate(category_names)}
        class_to_indices = [[] for _ in category_names]
        for i, category in enumerate(categories_list):
            class_to_indices[category_to_idx[category]].append(i)
        
        self.maira2_metrics = {}
        ious_list, cnrs_list, dice_list, soft_dice_list, precision_list, recall_list, rand_index_list = [], [], [], [], [], [], []
        
        for x in tqdm(maira2_predictions, desc="Calculating MAIRA-2 metrics"):
            gt_mask = convert_bboxes_into_presence_map(x['gt_bboxes'], (100, 100))
            pred_mask = convert_bboxes_into_presence_map(x['predicted_bboxes'], (100, 100))
            
            ious_list.append(calculate_segmentation_iou(gt_mask, pred_mask))
            cnrs_list.append(calculate_cnr(gt_mask, pred_mask))
            dice_list.append(calculate_dice(gt_mask, pred_mask))
            soft_dice_list.append(calculate_soft_dice(gt_mask, pred_mask))
            precision_list.append(calculate_segmentation_precision(gt_mask, pred_mask))
            recall_list.append(calculate_segmentation_recall(gt_mask, pred_mask))
            rand_index_list.append(calculate_rand_index(gt_mask, pred_mask))
        
        metric_lists = {'iou': ious_list, 'cnr': cnrs_list, 'dice': dice_list, 'soft_dice': soft_dice_list, 'precision': precision_list, 'recall': recall_list, 'rand_index': rand_index_list}
        for name, values in metric_lists.items():
            self.maira2_metrics[f'{name}_list'] = values
            out = apply_bootstrapping(metric_values=values, class_to_indices=class_to_indices, class_names=category_names, metric_name=name, use_tqdm=False)
            self.maira2_metrics.update(out)
        
        self.maira2_method_dict = {k: v['mean'] for k, v in self.maira2_metrics.items() if isinstance(v, dict)}
        self.maira2_method_dict.update({f"{k}_std": v['std'] for k, v in self.maira2_metrics.items() if isinstance(v, dict)})
        print("MAIRA-2 processing complete.")

    def _load_and_process_biovilt(self):
        """
        Loads BioViL-T predictions for VinDr-CXR and computes metrics.
        """
        print("Loading and processing BioViL-T predictions for VinDr-CXR...")
        biovilt_predictions = load_pickle(self.biovilt_preds_path)
        
        test_predictions = [x for x in biovilt_predictions if x['split'] == 'test']
        
        # In VinDr-CXR, the phrase itself is the category
        categories_list = [x['phrase'] for x in test_predictions]
        category_names = sorted(list(set(categories_list)))
        category_to_idx = {name: i for i, name in enumerate(category_names)}
        class_to_indices = [[] for _ in category_names]
        for i, category in enumerate(categories_list):
            class_to_indices[category_to_idx[category]].append(i)
            
        self._category_to_count = { category: len(class_to_indices[category_to_idx[category]]) for category in category_names }
        self._total_count = len(categories_list)
        
        pred_mask_list = [x['similarity_map'] for x in test_predictions]
        gt_bboxes_list = [x['gt_bboxes'] for x in test_predictions]

        if self.biovilt_threshold is None:
            print("Finding optimal threshold for BioViL-T...")
            sample_indices = random.sample(range(len(pred_mask_list)), min(50, len(pred_mask_list)))
            pred_mask_sample = [pred_mask_list[i] for i in sample_indices]
            gt_bboxes_sample = [gt_bboxes_list[i] for i in sample_indices]
            out = find_optimal_probability_map_conf_threshold(np.array(pred_mask_sample), gt_bboxes_sample)
            best_conf_th = out['best_conf_th']
        else:
            print(f"Using provided BioViL-T threshold: {self.biovilt_threshold}")
            best_conf_th = self.biovilt_threshold
        
        print(f"BioViL-T optimal threshold: {best_conf_th}")
        
        self.biovilt_metrics = {}
        ious_list, cnrs_list, dice_list, soft_dice_list, precision_list, recall_list, rand_index_list = [], [], [], [], [], [], []
        for pred_mask, gt_bboxes in tqdm(zip(pred_mask_list, gt_bboxes_list), total=len(pred_mask_list), desc="Calculating BioViL-T metrics"):
            gt_mask = convert_bboxes_into_presence_map(gt_bboxes, (100, 100))
            ious_list.append(calculate_segmentation_iou(gt_mask, pred_mask, threshold=best_conf_th))
            cnrs_list.append(calculate_cnr(gt_mask, pred_mask))
            dice_list.append(calculate_dice(gt_mask, pred_mask, threshold=best_conf_th))
            soft_dice_list.append(calculate_soft_dice(gt_mask, pred_mask))
            precision_list.append(calculate_segmentation_precision(gt_mask, pred_mask, threshold=best_conf_th))
            recall_list.append(calculate_segmentation_recall(gt_mask, pred_mask, threshold=best_conf_th))
            rand_index_list.append(calculate_rand_index(gt_mask, pred_mask, threshold=best_conf_th))
        
        metric_lists = {'iou': ious_list, 'cnr': cnrs_list, 'dice': dice_list, 'soft_dice': soft_dice_list, 'precision': precision_list, 'recall': recall_list, 'rand_index': rand_index_list}
        for name, values in metric_lists.items():
            self.biovilt_metrics[f'{name}_list'] = values
            out = apply_bootstrapping(metric_values=values, class_to_indices=class_to_indices, class_names=category_names, metric_name=name, use_tqdm=False)
            self.biovilt_metrics.update(out)
        
        self.biovilt_method_dict = {k: v['mean'] for k, v in self.biovilt_metrics.items() if isinstance(v, dict)}
        self.biovilt_method_dict.update({f"{k}_std": v['std'] for k, v in self.biovilt_metrics.items() if isinstance(v, dict)})
        print("BioViL-T processing complete.")

In [ ]:
# Define paths and aliases for own models
my_model_metrics_paths = [
    "/mnt/data/pamessina/workspaces/medvqa-workspace/results/phrase_grounding/20250817_143800_mscxr+chst-img-alg+chst-img-pg+vinbig+padchest-gr_PhraseGrounder(microsoft-rad-dino-maira-2,AdaptiveFiLM_MLP_BBoxRegression,128,256,256-128)/vindrcxr_predictions_and_gt.pkl.metrics.pkl",
    "/mnt/data/pamessina/workspaces/medvqa-workspace/results/phrase_grounding/20250727_172941_vinbig_PhraseGrounder(microsoft-rad-dino-maira-2,AdaptiveFiLM_MLP_BBoxRegression,128,256,256-128)/vindrcxr_predictions_and_gt.pkl.metrics.pkl",
    "/mnt/data/pamessina/workspaces/medvqa-workspace/results/phrase_grounding/20250804_141741_mscxr+chst-img-alg+chst-img-pg+vinbig+padchest-gr_PhraseGrounder(aehrc-cxrmate-rrg24-uniformer,AdaptiveFiLM_MLP_BBoxRegression,128,256,256-128)/vindrcxr_predictions_and_gt.pkl.metrics.pkl",
]
my_model_aliases = [
    "FG-VLM (RAD-DINO-MAIRA-2; 448x448; tr:MSCXR+PGR+Vin+CIG; val:MSCXR+PGR+Vin(SD))",
    "FG-VLM (RAD-DINO-MAIRA-2; 384x384; tr:Vin; val:Vin(CNR))",
    "FG-VLM (UniFormer; 384x384; tr:MSCXR+PGR+Vin+CIG; val:MSCXR+PGR+Vin(SD))",
]

# Paths for baselines
maira2_preds_path = '/home/pamessina/maira2_phrase_grounding_predictions/vindrcxr_phrase_grounding_results.jsonl'
biovilt_preds_path = '/home/pamessina/biovil-t_phrase_grounding_predictions/vindrcxr_phrase_grounding_results.pkl'

# Instantiate the comparator with the lists and a pre-computed threshold
vindrcxr_comparator = VinDrCXR_PhraseGroundingComparator(
    my_model_metrics_paths=my_model_metrics_paths,
    my_model_aliases=my_model_aliases,
    maira2_preds_path=maira2_preds_path,
    biovilt_preds_path=biovilt_preds_path,
    biovilt_threshold=0.6, # Pass the known threshold
)

In [ ]:
vindrcxr_comparator.plot_comparison(
    metric_key='prob_iou_with_bootstrapping',
    title='MS-CXR Phrase Grounding: Intersection over Union (IoU) based on probability maps',
    ylabel='IoU Score',
    sort_methods=True,
    sort_metrics=False,
    bbox_to_anchor=(0.605, 0.77),
    ylim=[0, 0.8],
    figsize=(12, 6),
    legend_fontsize=9.3,
)

In [ ]:
vindrcxr_comparator.plot_comparison(
    metric_key='dice_with_bootstrapping',
    title='MS-CXR Phrase Grounding: Dice',
    ylabel='Dice Score',
    sort_methods=True,
    sort_metrics=False,
    bbox_to_anchor=(0.605, 0.77),
    ylim=[0, 0.93],
    figsize=(12, 6),
    legend_fontsize=9.3,
)

# Metric plots for Chest ImaGenome

In [ ]:
import random
import numpy as np
from tqdm import tqdm

from medvqa.evaluation.bootstrapping import apply_bootstrapping
from medvqa.metrics.bbox.utils import compute_bbox_union_iou, find_optimal_probability_map_conf_threshold
from medvqa.utils.bbox_utils import convert_bboxes_into_presence_map
from medvqa.utils.files_utils import load_jsonl, load_pickle
from medvqa.utils.metrics_utils import (
    calculate_cnr, calculate_dice, calculate_rand_index, calculate_segmentation_iou,
    calculate_segmentation_precision, calculate_segmentation_recall, calculate_soft_dice
)

# --- New Class for ChestImaGenome ---

class ChestImaGenome_PhraseGroundingComparator(BasePhraseGroundingComparator):
    """
    Comparator for phrase grounding results on the ChestImaGenome dataset.
    """
    def __init__(self, **kwargs):
        super().__init__(**kwargs)

    def _load_and_process_maira2(self):
        """
        Loads MAIRA-2 predictions for ChestImaGenome and computes metrics.
        """
        print("Loading and processing MAIRA-2 predictions for ChestImaGenome...")
        maira2_predictions = load_jsonl(self.maira2_preds_path)
        
        # In ChestImaGenome, the phrase itself is the category
        categories_list = [x['phrase'] for x in maira2_predictions]
        category_names = sorted(list(set(categories_list)))
        category_to_idx = {name: i for i, name in enumerate(category_names)}
        class_to_indices = [[] for _ in category_names]
        for i, category in enumerate(categories_list):
            class_to_indices[category_to_idx[category]].append(i)
        
        self.maira2_metrics = {}
        ious_list, cnrs_list, dice_list, soft_dice_list, precision_list, recall_list, rand_index_list = [], [], [], [], [], [], []
        
        for x in tqdm(maira2_predictions, desc="Calculating MAIRA-2 metrics"):
            gt_bboxes = x['gt_bboxes']
            pred_bboxes = x['predicted_bboxes']
            gt_mask = convert_bboxes_into_presence_map(gt_bboxes, (100, 100))
            pred_mask = convert_bboxes_into_presence_map(pred_bboxes, (100, 100))
            
            ious_list.append(compute_bbox_union_iou(gt_bboxes, pred_bboxes))
            cnrs_list.append(calculate_cnr(gt_mask, pred_mask))
            dice_list.append(calculate_dice(gt_mask, pred_mask))
            soft_dice_list.append(calculate_soft_dice(gt_mask, pred_mask))
            precision_list.append(calculate_segmentation_precision(gt_mask, pred_mask))
            recall_list.append(calculate_segmentation_recall(gt_mask, pred_mask))
            rand_index_list.append(calculate_rand_index(gt_mask, pred_mask))
        
        metric_lists = {'iou': ious_list, 'cnr': cnrs_list, 'dice': dice_list, 'soft_dice': soft_dice_list, 'precision': precision_list, 'recall': recall_list, 'rand_index': rand_index_list}
        for name, values in metric_lists.items():
            self.maira2_metrics[f'{name}_list'] = values
            out = apply_bootstrapping(metric_values=values, class_to_indices=class_to_indices, class_names=category_names, metric_name=name, use_tqdm=False)
            self.maira2_metrics.update(out)
        
        self.maira2_method_dict = {k: v['mean'] for k, v in self.maira2_metrics.items() if isinstance(v, dict)}
        self.maira2_method_dict.update({f"{k}_std": v['std'] for k, v in self.maira2_metrics.items() if isinstance(v, dict)})
        print("MAIRA-2 processing complete.")

    def _load_and_process_biovilt(self):
        """
        Loads BioViL-T predictions for ChestImaGenome and computes metrics.
        """
        print("Loading and processing BioViL-T predictions for ChestImaGenome...")
        biovilt_predictions = load_pickle(self.biovilt_preds_path)
        
        test_predictions = [x for x in biovilt_predictions if x['split'] == 'test']
        
        # In ChestImaGenome, the phrase itself is the category
        categories_list = [x['phrase'] for x in test_predictions]
        category_names = sorted(list(set(categories_list)))
        category_to_idx = {name: i for i, name in enumerate(category_names)}
        class_to_indices = [[] for _ in category_names]
        for i, category in enumerate(categories_list):
            class_to_indices[category_to_idx[category]].append(i)
            
        self._category_to_count = { category: len(class_to_indices[category_to_idx[category]]) for category in category_names }
        self._total_count = len(categories_list)
        
        pred_mask_list = [x['similarity_map'] for x in test_predictions]
        gt_bboxes_list = [x['gt_bboxes'] for x in test_predictions]

        if self.biovilt_threshold is None:
            print("Finding optimal threshold for BioViL-T...")
            sample_indices = random.sample(range(len(pred_mask_list)), min(50, len(pred_mask_list)))
            pred_mask_sample = [pred_mask_list[i] for i in sample_indices]
            gt_bboxes_sample = [gt_bboxes_list[i] for i in sample_indices]
            out = find_optimal_probability_map_conf_threshold(np.array(pred_mask_sample), gt_bboxes_sample)
            best_conf_th = out['best_conf_th']
        else:
            print(f"Using provided BioViL-T threshold: {self.biovilt_threshold}")
            best_conf_th = self.biovilt_threshold
        
        print(f"BioViL-T optimal threshold: {best_conf_th}")
        
        self.biovilt_metrics = {}
        ious_list, cnrs_list, dice_list, soft_dice_list, precision_list, recall_list, rand_index_list = [], [], [], [], [], [], []
        for pred_mask, gt_bboxes in tqdm(zip(pred_mask_list, gt_bboxes_list), total=len(pred_mask_list), desc="Calculating BioViL-T metrics"):
            gt_mask = convert_bboxes_into_presence_map(gt_bboxes, (100, 100))
            ious_list.append(calculate_segmentation_iou(gt_mask, pred_mask, threshold=best_conf_th))
            cnrs_list.append(calculate_cnr(gt_mask, pred_mask))
            dice_list.append(calculate_dice(gt_mask, pred_mask, threshold=best_conf_th))
            soft_dice_list.append(calculate_soft_dice(gt_mask, pred_mask))
            precision_list.append(calculate_segmentation_precision(gt_mask, pred_mask, threshold=best_conf_th))
            recall_list.append(calculate_segmentation_recall(gt_mask, pred_mask, threshold=best_conf_th))
            rand_index_list.append(calculate_rand_index(gt_mask, pred_mask, threshold=best_conf_th))
        
        metric_lists = {'iou': ious_list, 'cnr': cnrs_list, 'dice': dice_list, 'soft_dice': soft_dice_list, 'precision': precision_list, 'recall': recall_list, 'rand_index': rand_index_list}
        for name, values in metric_lists.items():
            self.biovilt_metrics[f'{name}_list'] = values
            out = apply_bootstrapping(metric_values=values, class_to_indices=class_to_indices, class_names=category_names, metric_name=name, use_tqdm=False)
            self.biovilt_metrics.update(out)
        
        self.biovilt_method_dict = {k: v['mean'] for k, v in self.biovilt_metrics.items() if isinstance(v, dict)}
        self.biovilt_method_dict.update({f"{k}_std": v['std'] for k, v in self.biovilt_metrics.items() if isinstance(v, dict)})
        print("BioViL-T processing complete.")

In [ ]:
# Define paths and aliases for own models
my_model_metrics_paths = [
    "/mnt/data/pamessina/workspaces/medvqa-workspace/results/phrase_grounding/20250817_143800_mscxr+chst-img-alg+chst-img-pg+vinbig+padchest-gr_PhraseGrounder(microsoft-rad-dino-maira-2,AdaptiveFiLM_MLP_BBoxRegression,128,256,256-128)/chestimagenome_predictions_and_gt.pkl.metrics.pkl",
    "/mnt/data/pamessina/workspaces/medvqa-workspace/results/phrase_grounding/20250726_184906_chst-img-alg+chst-img-pg_PhraseGrounder(microsoft-rad-dino-maira-2,AdaptiveFiLM_MLP_BBoxRegression,128,256,256-128)/chestimagenome_predictions_and_gt.pkl.metrics.pkl",
    "/mnt/data/pamessina/workspaces/medvqa-workspace/results/phrase_grounding/20250614_180013_chst-img-alg+chst-img-pg_PhraseGrounder(aehrc-cxrmate-rrg24-uniformer,AdaptiveFiLM_MLP_BBoxRegression,128,256,256-128)/chestimagenome_predictions_and_gt.pkl.metrics.pkl",
]
my_model_aliases = [
    "FG-VLM (RAD-DINO-MAIRA-2; 448x448; tr:MSCXR+PGR+Vin+CIG; val:MSCXR+PGR+Vin(SD))",
    "FG-VLM (RAD-DINO-MAIRA-2; 384x384; tr:CIG; val:CIG(CNR))",
    "FG-VLM (UniFormer; 384x384; tr:CIG; val:CIG(CNR))",
]

# Paths for baselines
maira2_preds_path = '/home/pamessina/maira2_phrase_grounding_predictions/chestimagenome_phrase_grounding_results.jsonl'
biovilt_preds_path = '/home/pamessina/biovil-t_phrase_grounding_predictions/chest_imagenome_phrase_grounding_results.pkl'

# Instantiate the comparator with the lists and a pre-computed threshold
cig_comparator = ChestImaGenome_PhraseGroundingComparator(
    my_model_metrics_paths=my_model_metrics_paths,
    my_model_aliases=my_model_aliases,
    maira2_preds_path=maira2_preds_path,
    biovilt_preds_path=biovilt_preds_path,
    biovilt_threshold=0.6, # Pass the known threshold
)

In [ ]:
cig_comparator.plot_comparison(
    metric_key='prob_iou_with_bootstrapping',
    title='MS-CXR Phrase Grounding: Intersection over Union (IoU) based on probability maps',
    ylabel='IoU Score',
    sort_methods=True,
    sort_metrics=True,
    bbox_to_anchor=(0.65, 0.75),
    ylim=[0, 1.2],
    figsize=(14, 6),
    legend_fontsize=10,
)

In [ ]:
cig_comparator.plot_comparison(
    metric_key='bbox_iou_with_bootstrapping',
    title='MS-CXR Phrase Grounding: Intersection over Union (IoU) based on bounding boxes',
    ylabel='IoU Score',
    sort_methods=True,
    sort_metrics=True,
    bbox_to_anchor=(0.65, 0.75),
    ylim=[0, 1.2],
    figsize=(14, 6),
    legend_fontsize=10,
)

In [ ]:
cig_comparator.plot_comparison(
    metric_key='cnr_with_bootstrapping',
    title='MS-CXR Phrase Grounding: Intersection over Union (IoU) based on bounding boxes',
    ylabel='IoU Score',
    sort_methods=True,
    sort_metrics=True,
    bbox_to_anchor=(0.65, 0.75),
    ylim=[0, 8],
    figsize=(14, 6),
    legend_fontsize=10,
)

# Metric plots for ChestX-Det

In [ ]:
import os
import random
import numpy as np
from tqdm import tqdm

# Assuming these imports are available in your environment
from medvqa.evaluation.bootstrapping import apply_bootstrapping
from medvqa.utils.bbox_utils import convert_bboxes_into_presence_map
from medvqa.utils.files_utils import load_jsonl, load_pickle
from medvqa.utils.metrics_utils import (
    calculate_cnr, calculate_dice, calculate_rand_index, calculate_segmentation_iou,
    calculate_segmentation_precision, calculate_segmentation_recall, calculate_soft_dice,
    find_optimal_probability_map_conf_threshold_for_masks
)
from medvqa.datasets.chestxdet.chestxdet_phrase_grounding_dataset_management import polygons_to_mask

# --- New Class for ChestX-Det with Per-Class Bootstrapping ---

class ChestXDet_PhraseGroundingComparator(BasePhraseGroundingComparator):
    """
    Comparator for phrase grounding results on the ChestX-Det dataset.
    This class builds a ground-truth polygon map from a source file to
    evaluate baseline models with per-class bootstrapping.
    """
    def __init__(self, **kwargs):
        # Build the ground-truth polygon map before initializing the parent
        self._build_polygon_map(kwargs['my_model_metrics_paths'])
        super().__init__(**kwargs)

    def _build_polygon_map(self, my_model_metrics_paths):
        """
        Builds a map from (image_filename, phrase) to ground-truth polygons
        using one of the provided model prediction files.
        """
        print("Building ChestX-Det ground-truth polygon map...")
        if not my_model_metrics_paths:
            raise ValueError("Cannot build polygon map without at least one custom model path.")
        
        # Derive the prediction file path from the first metrics file path
        gt_source_path = my_model_metrics_paths[0].replace('.metrics.pkl', '')
        print(f"Loading ground-truth data from: {gt_source_path}")
        
        gt_data = load_pickle(gt_source_path)['test_preds_and_gt']
        image_paths = gt_data['image_paths']
        phrases = gt_data['phrases']
        gt_polygons_list = gt_data['gt_polygons']
        
        self.image_phrase_to_polygons = {}
        for img_path, phrase, polygons in zip(image_paths, phrases, gt_polygons_list):
            # Extract filename without extension, e.g., '36302'
            filename = os.path.splitext(os.path.basename(img_path))[0]
            self.image_phrase_to_polygons[(filename, phrase)] = polygons
        
        print(f"Polygon map built with {len(self.image_phrase_to_polygons)} entries.")

    def _load_and_process_maira2(self):
        """
        Loads MAIRA-2 predictions for ChestX-Det and computes metrics with
        per-class bootstrapping.
        """
        print("Loading and processing MAIRA-2 predictions for ChestX-Det...")
        maira2_predictions = load_jsonl(self.maira2_preds_path)
        
        # --- NEW: Prepare for per-class bootstrapping ---
        categories_list = [x['phrase'] for x in maira2_predictions]
        category_names = sorted(list(set(categories_list)))
        category_to_idx = {name: i for i, name in enumerate(category_names)}
        class_to_indices = [[] for _ in category_names]
        for i, category in enumerate(categories_list):
            class_to_indices[category_to_idx[category]].append(i)
        
        self.maira2_metrics = {}
        ious_list, cnrs_list, dice_list, soft_dice_list, precision_list, recall_list, rand_index_list = [], [], [], [], [], [], []
        
        for x in tqdm(maira2_predictions, desc="Calculating MAIRA-2 metrics"):
            filename = os.path.splitext(x['file_name'])[0]
            phrase = x['phrase']
            gt_polygons = self.image_phrase_to_polygons[(filename, phrase)]
            gt_mask = polygons_to_mask(gt_polygons, 200, 200)
            pred_mask = convert_bboxes_into_presence_map(x['predicted_bboxes'], (200, 200))
            
            ious_list.append(calculate_segmentation_iou(gt_mask, pred_mask))
            cnrs_list.append(calculate_cnr(gt_mask, pred_mask))
            dice_list.append(calculate_dice(gt_mask, pred_mask))
            soft_dice_list.append(calculate_soft_dice(gt_mask, pred_mask))
            precision_list.append(calculate_segmentation_precision(gt_mask, pred_mask))
            recall_list.append(calculate_segmentation_recall(gt_mask, pred_mask))
            rand_index_list.append(calculate_rand_index(gt_mask, pred_mask))

        metric_lists = {'iou': ious_list, 'cnr': cnrs_list, 'dice': dice_list, 'soft_dice': soft_dice_list, 'precision': precision_list, 'recall': recall_list, 'rand_index': rand_index_list}
        
        for name, values in metric_lists.items():
            self.maira2_metrics[f'{name}_list'] = values
            # --- UPDATED: Apply bootstrapping with class info ---
            out = apply_bootstrapping(
                metric_values=values,
                class_to_indices=class_to_indices,
                class_names=category_names,
                metric_name=name,
                use_tqdm=False
            )
            self.maira2_metrics.update(out)
        
        # --- UPDATED: Create method dict from nested results ---
        self.maira2_method_dict = {k: v['mean'] for k, v in self.maira2_metrics.items() if isinstance(v, dict)}
        self.maira2_method_dict.update({f"{k}_std": v['std'] for k, v in self.maira2_metrics.items() if isinstance(v, dict)})
        print("MAIRA-2 processing complete.")

    def _load_and_process_biovilt(self):
        """
        Loads BioViL-T predictions for ChestX-Det and computes metrics with
        per-class bootstrapping.
        """
        print("Loading and processing BioViL-T predictions for ChestX-Det...")
        biovilt_predictions = load_pickle(self.biovilt_preds_path)
        
        test_predictions = [x for x in biovilt_predictions if x['split'] == 'test']
        
        # --- NEW: Prepare for per-class bootstrapping ---
        categories_list = [x['phrase'] for x in test_predictions]
        category_names = sorted(list(set(categories_list)))
        category_to_idx = {name: i for i, name in enumerate(category_names)}
        class_to_indices = [[] for _ in category_names]
        for i, category in enumerate(categories_list):
            class_to_indices[category_to_idx[category]].append(i)
            
        self._category_to_count = { category: len(class_to_indices[category_to_idx[category]]) for category in category_names }
        self._total_count = len(categories_list)

        pred_mask_list = []
        gt_mask_list = []
        for x in tqdm(test_predictions, desc="Preparing BioViL-T GT masks"):
            pred_mask_list.append(x['similarity_map'])
            filename = os.path.splitext(os.path.basename(x['image_path']))[0]
            phrase = x['phrase']
            gt_polygons = self.image_phrase_to_polygons[(filename, phrase)]
            gt_mask_list.append(polygons_to_mask(gt_polygons, 100, 100))

        if self.biovilt_threshold is None:
            print("Finding optimal threshold for BioViL-T...")
            sample_indices = random.sample(range(len(pred_mask_list)), min(500, len(pred_mask_list)))
            pred_mask_sample = [pred_mask_list[i] for i in sample_indices]
            gt_mask_sample = [gt_mask_list[i] for i in sample_indices]
            
            out = find_optimal_probability_map_conf_threshold_for_masks(
                np.array(pred_mask_sample), np.array(gt_mask_sample)
            )
            best_conf_th = out['best_conf_th']
        else:
            print(f"Using provided BioViL-T threshold: {self.biovilt_threshold}")
            best_conf_th = self.biovilt_threshold
        
        print(f"BioViL-T optimal threshold: {best_conf_th}")
        
        self.biovilt_metrics = {}
        ious_list, cnrs_list, dice_list, soft_dice_list, precision_list, recall_list, rand_index_list = [], [], [], [], [], [], []
        
        for pred_mask, gt_mask in tqdm(zip(pred_mask_list, gt_mask_list), total=len(pred_mask_list), desc="Calculating BioViL-T metrics"):
            ious_list.append(calculate_segmentation_iou(gt_mask, pred_mask, threshold=best_conf_th))
            cnrs_list.append(calculate_cnr(gt_mask, pred_mask))
            dice_list.append(calculate_dice(gt_mask, pred_mask, threshold=best_conf_th))
            soft_dice_list.append(calculate_soft_dice(gt_mask, pred_mask))
            precision_list.append(calculate_segmentation_precision(gt_mask, pred_mask, threshold=best_conf_th))
            recall_list.append(calculate_segmentation_recall(gt_mask, pred_mask, threshold=best_conf_th))
            rand_index_list.append(calculate_rand_index(gt_mask, pred_mask, threshold=best_conf_th))

        metric_lists = {'iou': ious_list, 'cnr': cnrs_list, 'dice': dice_list, 'soft_dice': soft_dice_list, 'precision': precision_list, 'recall': recall_list, 'rand_index': rand_index_list}
        
        for name, values in metric_lists.items():
            self.biovilt_metrics[f'{name}_list'] = values
            # --- UPDATED: Apply bootstrapping with class info ---
            out = apply_bootstrapping(
                metric_values=values,
                class_to_indices=class_to_indices,
                class_names=category_names,
                metric_name=name,
                use_tqdm=False
            )
            self.biovilt_metrics.update(out)
            
        # --- UPDATED: Create method dict from nested results ---
        self.biovilt_method_dict = {k: v['mean'] for k, v in self.biovilt_metrics.items() if isinstance(v, dict)}
        self.biovilt_method_dict.update({f"{k}_std": v['std'] for k, v in self.biovilt_metrics.items() if isinstance(v, dict)})
        print("BioViL-T processing complete.")

In [ ]:
# Define paths and aliases for own models
my_model_metrics_paths = [
    "/mnt/data/pamessina/workspaces/medvqa-workspace/results/phrase_grounding/20250817_143800_mscxr+chst-img-alg+chst-img-pg+vinbig+padchest-gr_PhraseGrounder(microsoft-rad-dino-maira-2,AdaptiveFiLM_MLP_BBoxRegression,128,256,256-128)/chestxdet_predictions_and_gt.pkl.metrics.pkl",
    "/mnt/data/pamessina/workspaces/medvqa-workspace/results/phrase_grounding/20250727_172941_vinbig_PhraseGrounder(microsoft-rad-dino-maira-2,AdaptiveFiLM_MLP_BBoxRegression,128,256,256-128)/chestxdet_predictions_and_gt.pkl.metrics.pkl",
    "/mnt/data/pamessina/workspaces/medvqa-workspace/results/phrase_grounding/20250804_141741_mscxr+chst-img-alg+chst-img-pg+vinbig+padchest-gr_PhraseGrounder(aehrc-cxrmate-rrg24-uniformer,AdaptiveFiLM_MLP_BBoxRegression,128,256,256-128)/chestxdet_predictions_and_gt.pkl.metrics.pkl",
]
my_model_aliases = [
    "FG-VLM (RAD-DINO-MAIRA-2; 448x448; tr:MSCXR+PGR+Vin+CIG; val:MSCXR+PGR+Vin(SD))",
    "FG-VLM (RAD-DINO-MAIRA-2; 384x384; tr:Vin; val:Vin(CNR))",
    "FG-VLM (UniFormer; 384x384; tr:MSCXR+PGR+Vin+CIG; val:MSCXR+PGR+Vin(SD))",
]

# Paths for baselines
maira2_preds_path = '/home/pamessina/maira2_phrase_grounding_predictions/chestxdet_phrase_grounding_results.jsonl'
biovilt_preds_path = '/home/pamessina/biovil-t_phrase_grounding_predictions/chestxdet_phrase_grounding_results.pkl'

# Instantiate the comparator with the lists and a pre-computed threshold
chestxdet_comparator = ChestXDet_PhraseGroundingComparator(
    my_model_metrics_paths=my_model_metrics_paths,
    my_model_aliases=my_model_aliases,
    maira2_preds_path=maira2_preds_path,
    biovilt_preds_path=biovilt_preds_path,
    biovilt_threshold=0.6,
)

In [ ]:
chestxdet_comparator.plot_comparison(
    metric_key='prob_iou_with_bootstrapping',
    title='MS-CXR Phrase Grounding: Intersection over Union (IoU) based on probability maps',
    ylabel='IoU Score',
    sort_methods=True,
    sort_metrics=False,
    bbox_to_anchor=(0.63, 0.77),
    ylim=[0, 0.72],
    figsize=(12, 6),
    legend_fontsize=9,
)

In [ ]:
chestxdet_comparator.plot_comparison(
    metric_key='dice_with_bootstrapping',
    title='MS-CXR Phrase Grounding: Dice',
    ylabel='Dice IoU Score',
    sort_methods=True,
    sort_metrics=False,
    bbox_to_anchor=(0.63, 0.77),
    ylim=[0, 0.9],
    figsize=(12, 6),
    legend_fontsize=9,
)

In [ ]:
chestxdet_comparator.plot_comparison(
    metric_key='cnr_with_bootstrapping',
    title='MS-CXR Phrase Grounding: CNR',
    ylabel='CNR',
    sort_methods=True,
    sort_metrics=False,
    bbox_to_anchor=(0.63, 0.77),
    ylim=[-0.15, 4.5],
    figsize=(12, 6),
    legend_fontsize=9,
)